In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import numpy as np

In [2]:
from pathlib import Path

# Prefer the full dataset if it has been fetched (see ../fetch_data.py);
# otherwise use the stratified 20% sample committed with the repository, which
# preserves the class distribution exactly (84.2 / 1.8 / 13.9).
DATA = Path.cwd().parent / "data"
_full, _sample = DATA / "diabetes_brfss2015.csv", DATA / "diabetes_brfss2015_sample.csv"
file_path = _full if _full.exists() else _sample
print(f"using {file_path.name}")
df = pd.read_csv(file_path)
print(f"{len(df):,} rows x {df.shape[1]} columns")
df.head()


using diabetes_brfss2015_sample.csv
50,736 rows x 22 columns


,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,0.0,0.0,1.0,24.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,4.0,0.0,15.0,1.0,0.0,9.0,5.0,5.0
1,0.0,1.0,0.0,1.0,32.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,3.0,0.0,0.0,0.0,0.0,8.0,6.0,8.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,3.0,1.0,0.0,0.0,0.0,9.0,6.0,6.0
3,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,0.0,0.0,1.0,9.0,6.0,8.0
4,0.0,0.0,0.0,1.0,35.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,3.0,10.0,2.0,0.0,0.0,6.0,4.0,8.0


In [3]:
# ===============================
# 2. Jika ingin drop kelas 1
# ===============================
df = df[df["Diabetes_012"] != 1]

# ===============================
# 3. Split fitur & target
# ===============================
X = df.drop("Diabetes_012", axis=1)
y = df["Diabetes_012"]

In [4]:
# ===============================
# 4. Train-test split
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ===============================
# 5. Hyperparameter Grid
# ===============================
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "bootstrap": [True, False]
}

rf = RandomForestClassifier(random_state=42)

grid_search = GridSearchCV(
    rf,
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

# ===============================
# 6. Fit model
# ===============================
grid_search.fit(X_train, y_train)

print("Best Params:", grid_search.best_params_)
print("Best CV Score:", grid_search.best_score_)

# ===============================
# 7. Evaluate best model
# ===============================
best_rf = grid_search.best_estimator_

y_pred = best_rf.predict(X_test)

print("\nTest Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Fitting 5 folds for each of 216 candidates, totalling 1080 fits


Best Params: {'bootstrap': True, 'max_depth': 30, 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 300}
Best CV Score: 0.8663922165140232



Test Accuracy: 0.86197550692632

Confusion Matrix:
 [[8360  188]
 [1187  227]]

Classification Report:
               precision    recall  f1-score   support

         0.0       0.88      0.98      0.92      8548
         2.0       0.55      0.16      0.25      1414

    accuracy                           0.86      9962
   macro avg       0.71      0.57      0.59      9962
weighted avg       0.83      0.86      0.83      9962

